# Custom Configuration Pipeline

Build fully customized pipelines from scratch with complete control over preprocessing strategies, feature engineering components, and optimization parameters.

In [ ]:
from featransform.pipeline import Featransform
from featransform.core.models import PipelineConfig, ProcessingConfig, OptimizationConfig, ModelConfig
from featransform.core.enums import ImputationStrategy, EncodingStrategy, SelectionStrategy, ModelFamily
from featransform.utils.data_generator import DatasetGenerator
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore", category=Warning)

## Generate Dataset

In [ ]:
X, y = DatasetGenerator.generate(
    task='multiclass_classification',
    n_samples=5000,
    n_features=20,
    n_informative=15,
    add_datetime=True,
    n_datetime_cols=2,
    add_categorical=True,
    n_categorical=3,
    add_missing=True,
    missing_rate=0.1,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Custom Configuration

In [ ]:
config = PipelineConfig(
    task_type="multiclass_classification",
    processing=ProcessingConfig(
        imputation_strategy=ImputationStrategy.ITERATIVE,
        encoding_strategy=EncodingStrategy.LABEL,
        handle_datetime=True,
        drop_constant=True,
        drop_duplicates=True
    ),
    anomaly_models=[
        ModelConfig(model_family=ModelFamily.ISOLATION_FOREST, parameters={'n_estimators': 300, 'contamination': 0.002}),
        ModelConfig(model_family=ModelFamily.LOCAL_OUTLIER_FACTOR, parameters={'n_neighbors': 20, 'contamination': 0.002})
    ],
    clustering_models=[
        ModelConfig(model_family=ModelFamily.KMEANS, parameters={'n_clusters': 3}),
        ModelConfig(model_family=ModelFamily.BIRCH, parameters={'threshold': 0.5, 'branching_factor': 50})
    ],
    dimensionality_models=[
        ModelConfig(model_family=ModelFamily.PCA, parameters={'n_components': 0.95}),
        ModelConfig(model_family=ModelFamily.FAST_ICA, parameters={'n_components': 6})
    ],
    optimization=OptimizationConfig(
        selection_strategy=SelectionStrategy.IMPORTANCE,
        n_iterations=10,
        validation_split=0.3,
        min_features=10
    ),
    verbose=True,
    n_jobs=-1,
    random_state=42
)

## Fit & Transform

In [ ]:
ft = Featransform(config)
ft.fit(X_train, y_train)
X_train_transformed = ft.transform(X_train)
X_test_transformed = ft.transform(X_test)

## Results

In [ ]:
ft.report_optimization()